# Live parallax in a notebook

The notebook is a **subscriber** to the pipeline's live stream: `run.py` publishes a snapshot of the
reconstruction-so-far every `--live-every` seconds (here: the DC-corrected parallax image and its
phase-flipped counterpart), and `LiveView` draws each one as it arrives. Updates run on the
kernel's event loop, so the notebook stays usable while the scan fills in.

* **At the microscope** the pipeline runs on the GPU machine (`run.py ... --live-endpoint
  tcp://0.0.0.0:5601`); point `ENDPOINT` at it and skip the "start the pipeline" cell.
* **Here** we replay a simulated 256×256 scan (96×96 frames) through the same pipeline in a
  subprocess, with the pure-Python ingest, at 8 kHz so the fill-in is watchable.

In [ ]:
# %matplotlib widget
# works too (the output cell is re-rendered on each update)
%matplotlib inline

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
sys.path.insert(0, str(REPO))        # not needed once the package is pip-installed

ENDPOINT = "tcp://127.0.0.1:5601"    # at the microscope: "tcp://<detector-box>:5601"
CAPTURE = REPO / "out" / "sim256.dstream"
OUT = REPO / "out" / "notebook"

if not CAPTURE.exists():             # ~1 min: simulate the scan (writes .dstream + .flags + truth)
    subprocess.run([sys.executable, "-m", "tools.simulate_4dstem", "--scan", "256", "256",
                    "--format", "npy", "dstream", "--out", str(CAPTURE.with_suffix(""))],
                   cwd=REPO, check=True)

## Subscribe

Start the view *before* the pipeline publishes (a PUB/SUB subscriber only sees frames sent after
it connects).

In [ ]:
from realtime_stem.streaming.notebook import LiveView

view = LiveView(ENDPOINT, panel_size=4.0)
view.start()
view

## Start the pipeline (local replay)

Exactly the command you would run in a terminal; `.flags` holds the simulated geometry
(`--defocus`, `--rotation`, samplings, …). `--lowpass 0.1` (1/Å) applies a Butterworth low-pass to
the live channels, which helps at this dose.

In [ ]:
flags = CAPTURE.with_suffix(".flags").read_text().split()
pipeline = subprocess.Popen(
    [sys.executable, "-m", "realtime_stem.apps.run", "--source", str(CAPTURE), "--source-rate", "8000",
     "--batch-size", "256", "--live-endpoint", ENDPOINT, "--live-every", "0.25", "--lowpass", "0.1",
     "--out-dir", str(OUT), *flags],
    cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

The view above updates while the scan runs. Awaiting the final frame is optional — it just makes
the cells below wait for the end of the scan:

In [ ]:
arr, header = await view.wait_final(timeout=300)
log = pipeline.communicate()[0]
print("\n".join(line for line in log.splitlines() if "TOTAL" in line or "saved" in line))
print(f"{view.n_updates} live updates received")

## Mid-acquisition snapshots

`view.history` keeps the frames it received; `snapshot_at` picks the one closest to a scanned
fraction.

In [ ]:
import matplotlib.pyplot as plt
from realtime_stem.streaming.display import robust_clim

fractions = (0.25, 0.5, 0.75, 1.0)
with plt.ioff():
    fig, axs = plt.subplots(2, len(fractions), figsize=(3.2 * len(fractions), 6.8))
for col, frac in enumerate(fractions):
    stack, hdr = view.snapshot_at(frac)
    for row, name in enumerate(hdr["channels"]):
        img = stack[row]
        scanned = img[: max(8, int(hdr["meta"]["coverage"] * img.shape[0]) - 12)]   # clim from the scanned part
        lo, hi = robust_clim(scanned)
        axs[row, col].imshow(img, cmap="gray", vmin=lo, vmax=hi)
        axs[row, col].set_title(f"{name}    {100 * hdr['meta']['coverage']:.0f}% scanned", fontsize=9)
        axs[row, col].axis("off")
fig.tight_layout()
fig

In [ ]:
view.save_gif(OUT / "live.gif", fps=6)   # the same frames, as an animation
print(OUT / "live.gif")

## Compare with the simulated ground truth

At negative C10 the raw parallax image has inverted contrast at low spatial frequencies; the phase
flip restores the object's sign (bright shells, like the truth).

In [ ]:
import numpy as np

truth = np.load(CAPTURE.with_name(CAPTURE.stem + "_phase.npy"))
images = {"ground-truth phase": truth,
          "parallax (corrected)": np.load(OUT / "parallax_corrected.npy"),
          "phase flipped": np.load(OUT / "parallax_phase_flipped.npy")}
with plt.ioff():
    fig, axs = plt.subplots(1, 3, figsize=(12, 4.3))
for ax, (name, img) in zip(axs, images.items()):
    lo, hi = robust_clim(img)
    ax.imshow(img, cmap="gray", vmin=lo, vmax=hi)
    ax.set_title(name)
    ax.axis("off")
fig.tight_layout()
fig

In [ ]:
view.stop()